# ITCS 6169/8169 — Computer Vision
## One-Shot Image Classification Experimental Template (Assignment 3)

A template following the original notebook's organization: imports, project paths, reproducibility, preprocessing, dataset composition and visualization, model, training utilities, experiment tracking, and final evaluation.

**Assignment 3 goal:** the highest possible test accuracy on the 16-class dataset using **exactly one labeled training image per class (16 images in total)**, built on foundation models such as CLIP, SigLIP, or DINOv2. The report (max 2 pages) is due **November 24, 2026, 11:59 PM ET**.

**One-shot warning:** using more than one labeled training image per class is strictly forbidden. This notebook reads training labels in exactly one place, `build_support_set()`, which asserts one image per class. There is no labeled validation set, so anything you choose by looking at test accuracy is tuning on the test set. Keep the search small and disclose it in your report. Unlabeled training images are off by default (`USE_UNLABELED_TRAIN = False`); check with your instructor before enabling them.

In [ ]:
# Core imports
from pathlib import Path
import random, time, copy, math, gc, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from IPython.display import display

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Run once (Colab / fresh environment). SigLIP's tokenizer needs sentencepiece + protobuf.
%pip install -q transformers sentencepiece protobuf

In [ ]:
# Optional Google Drive mount for Colab
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab.")

## 1. Set your working directory

Expected dataset structure:

```text
data/
  train/
    class_1/
    class_2/
  test/
    class_1/
    class_2/
```

In [ ]:
# Change this path to your project directory.
# PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169_8169/assignment3')
PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print("Project root:", PROJECT_ROOT.resolve())
print("Training data:", TRAIN_ROOT)
print("Testing data:", TEST_ROOT)
assert TRAIN_ROOT.exists(), "Training directory not found."
assert TEST_ROOT.exists(), "Testing directory not found."

## 2. Reproducibility and device

In [ ]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

## 3. Loading and preprocessing the data

The dataset is composed in three parts: the **one-shot support set** (the only labeled training images), an **augmented support dataset** (many views of those same 16 images), and the **test set**. Images stay as RGB PIL images because each foundation model's own processor handles resizing and normalization, so no image-size or patch-size settings are needed here.

In [ ]:
# One-shot data settings
SUPPORT_SEED = 0              # which single image per class is the labeled sample
SEEDS = [0, 1, 2, 3, 4]       # support sets to average over (shows sensitivity to the chosen image)
N_AUG_VIEWS = 32              # views per support image (view 0 is the original)
USE_UNLABELED_TRAIN = False   # OFF by default: check with your instructor before enabling

#############################################################################
# Data augmentation: more views of the SAME single labeled image
# Adjust augmentation as part of your experiments.
support_transform = transforms.Compose([
    transforms.RandomResizedCrop(256, scale=(0.4, 1.0), ratio=(0.75, 1.333)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.2, 0.02)], p=0.8),
    transforms.RandomApply([transforms.GaussianBlur(9, sigma=(0.1, 1.5))], p=0.3),
    transforms.RandomGrayscale(p=0.1),
])


def make_views(image, n_views, seed=0):
    """Return [original] + (n_views - 1) random augmentations (deterministic for a given seed)."""
    state = torch.get_rng_state()
    torch.manual_seed(seed)
    views = [image] + [support_transform(image) for _ in range(n_views - 1)]
    torch.set_rng_state(state)
    return views

###############################################################################################
# Initialize dataset configurations (PIL images, no transforms)
train_folder = datasets.ImageFolder(TRAIN_ROOT)
test_dataset = datasets.ImageFolder(TEST_ROOT)
class_names = train_folder.classes
num_classes = len(class_names)

assert test_dataset.class_to_idx == train_folder.class_to_idx, (
    "Train and test class mappings do not match."
)

test_paths = [p for p, _ in test_dataset.samples]
test_labels = torch.tensor(test_dataset.targets)

###############################################################################################
# One-shot support set: the ONLY place training labels are used.

def select_support_indices(folder, seed):
    """Pick one random image per class."""
    rng = np.random.RandomState(seed)
    targets = np.asarray(folder.targets)
    return [int(rng.choice(np.where(targets == c)[0])) for c in range(len(folder.classes))]


def build_support_set(folder, seed):
    """Return (images, labels, file paths, indices): exactly one image per class, ordered by class id."""
    idx = select_support_indices(folder, seed)
    images = [folder[i][0] for i in idx]
    labels = [folder.targets[i] for i in idx]
    files = [folder.samples[i][0] for i in idx]
    # Hard one-shot guard: all distinct, exactly one per class
    assert len(idx) == len(set(idx)) == len(folder.classes), "Support set must have one image per class."
    assert labels == list(range(len(folder.classes))), "Support set must contain each class exactly once."
    return images, torch.tensor(labels), files, idx


class OneShotSupportDataset(torch.utils.data.Dataset):
    """The 16 labeled images, each expanded into n_views views (class-major order)."""
    def __init__(self, images, labels, n_views, seed):
        self.views, self.labels = [], []
        for k, (img, lab) in enumerate(zip(images, labels)):
            for v in make_views(img, n_views, seed + k):
                self.views.append(v)
                self.labels.append(int(lab))

    def __len__(self):
        return len(self.views)

    def __getitem__(self, idx):
        return self.views[idx], self.labels[idx]


support_images, support_labels, support_files, _ = build_support_set(train_folder, SUPPORT_SEED)
support_dataset = OneShotSupportDataset(support_images, support_labels.tolist(), N_AUG_VIEWS, SUPPORT_SEED)

# Optional unlabeled pool (paths only; labels are never read). Disabled by default.
unlabeled_paths = [p for p, _ in train_folder.samples] if USE_UNLABELED_TRAIN else []

print("Classes:", class_names)
print(f"Labeled support images: {len(support_images)} | Augmented support views: {len(support_dataset)} | "
      f"Test: {len(test_dataset)}")
print(f"Training images on disk (labels NOT used): {len(train_folder)} | "
      f"Unlabeled training images used: {len(unlabeled_paths)}")

### Dataset summary tables

The functions below display per-class counts and total image counts for each split.

In [ ]:
def display_dataset_tables(support_labels, augmented_dataset, train_folder, test_dataset, class_names):
    """Display per-class and per-split image counts as pandas tables."""
    support_targets = np.asarray(support_labels)
    view_targets = np.asarray(augmented_dataset.labels)
    train_targets = np.asarray(train_folder.targets)
    test_targets = np.asarray(test_dataset.targets)

    rows = []
    for class_id, class_name in enumerate(class_names):
        rows.append({
            "Class": class_name,
            "Labeled support": int(np.sum(support_targets == class_id)),
            "Augmented views": int(np.sum(view_targets == class_id)),
            "Train on disk (unused)": int(np.sum(train_targets == class_id)),
            "Test": int(np.sum(test_targets == class_id)),
        })

    class_table = pd.DataFrame(rows)
    totals = {"Class": "TOTAL"}
    for col in class_table.columns[1:]:
        totals[col] = int(class_table[col].sum())
    class_table.loc[len(class_table)] = totals

    split_table = pd.DataFrame([
        {"Split": "Labeled support (used)", "Images": len(support_targets)},
        {"Split": "Augmented views of support", "Images": len(view_targets)},
        {"Split": "Train on disk (labels unused)", "Images": len(train_targets)},
        {"Split": "Test", "Images": len(test_targets)},
    ])

    print("Images per class and split")
    display(class_table)
    print("Dataset split summary")
    display(split_table)
    return class_table, split_table


class_count_table, split_summary_table = display_dataset_tables(
    support_labels, support_dataset, train_folder, test_dataset, class_names
)

### Visualize training examples

The first figure shows the 16 labeled images. The second shows the original and augmented views of one of them.

In [ ]:
def show_support_set(images, labels, class_names):
    cols = 4
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = np.asarray(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, image, label in zip(axes, images, labels):
        ax.imshow(image)
        ax.set_title(class_names[int(label)])
        ax.axis("off")

    plt.suptitle(f"The {len(images)} labeled training images (support seed {SUPPORT_SEED})")
    plt.tight_layout()
    plt.show()


def show_augmented_views(dataset, class_id, class_names, n_views=8, n_total=N_AUG_VIEWS):
    start = class_id * n_total
    cols = 4
    rows = math.ceil(n_views / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = np.asarray(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for k, ax in enumerate(axes[:n_views]):
        image, _ = dataset[start + k]
        ax.imshow(image)
        ax.set_title("original" if k == 0 else f"view {k}")
        ax.axis("off")

    plt.suptitle(f"Views of the single '{class_names[class_id]}' support image")
    plt.tight_layout()
    plt.show()


show_support_set(support_images, support_labels, class_names)
show_augmented_views(support_dataset, class_id=0, class_names=class_names)

## 4. Foundation model and one-shot classifier

The model has two parts:

- **Frozen foundation-model backbone** (`FoundationModel`): CLIP, SigLIP, or DINOv2 turns an image (or a text prompt) into an L2-normalized embedding. It is never trained, so the 16 labeled images can't overfit it.
- **Trainable prototype head** (`PrototypeHead`): one weight vector per class, compared with image embeddings by cosine similarity. It starts from the zero-shot text prototypes fused with the support-image prototypes and is trained on augmented support views, with an L2 term that keeps it near its starting point.

The cell below writes `oneshot_lib.py` next to the notebook so that `inference.py` in your GitHub repo imports exactly the same code. If you edit it, restart the kernel or run `importlib.reload(oneshot_lib)`. It also contains the **test-time** pieces: test-time augmentation and transductive refinement (soft k-means on unlabeled test features, with class-balanced Sinkhorn assignments, anchored to the prototypes).

In [ ]:
%%writefile oneshot_lib.py
"""Foundation-model backbone, prototype head and test-time adaptation for one-shot classification."""
import re
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T

# ----------------------------------------------------------------------------
# Text prompts
# ----------------------------------------------------------------------------
PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a photo of the {}.",
    "a picture of a {}.",
    "a low resolution photo of a {}.",
    "a {} scene.",
    "an image showing a {}.",
]

# Human-readable phrasing for class folder names. Edit to match your class list.
DEFAULT_DESCRIPTIONS = {
    "Bedroom": "bedroom",
    "Coast": "coast with ocean and beach",
    "Flower": "flower",
    "Forest": "forest",
    "Highway": "highway road",
    "Industrial": "industrial building site",
    "InsideCity": "downtown city street",
    "Kitchen": "kitchen",
    "LivingRoom": "living room",
    "Mountain": "mountain landscape",
    "Office": "office interior",
    "OpenCountry": "open countryside field",
    "Store": "store interior with shelves",
    "Street": "city street",
    "Suburb": "suburban neighborhood with houses",
    "TallBuilding": "skyscraper",
}


def class_to_text(name, overrides=None):
    overrides = DEFAULT_DESCRIPTIONS if overrides is None else overrides
    if name in overrides:
        return overrides[name]
    return re.sub(r"(?<!^)(?=[A-Z])", " ", name).replace("_", " ").lower()


def _load(x):
    if isinstance(x, (str, Path)):
        return Image.open(x).convert("RGB")
    return x.convert("RGB") if x.mode != "RGB" else x


def _as_tensor(x):
    if torch.is_tensor(x):
        return x
    for attr in ("pooler_output", "image_embeds", "text_embeds"):
        v = getattr(x, attr, None)
        if v is not None:
            return v
    return x[0]


# ----------------------------------------------------------------------------
# Frozen backbone
# ----------------------------------------------------------------------------
class FoundationModel:
    """Frozen backbone that returns L2-normalized image (and text) embeddings."""

    def __init__(self, kind, hf_name, device):
        from transformers import AutoModel, AutoProcessor, AutoTokenizer
        self.kind, self.hf_name, self.device = kind, hf_name, device
        model = AutoModel.from_pretrained(hf_name)
        if device.type == "cuda":
            model = model.half()
        self.model = model.to(device).eval()
        self.dtype = next(self.model.parameters()).dtype
        self.processor = AutoProcessor.from_pretrained(hf_name)
        self.tokenizer = None if kind == "dinov2" else AutoTokenizer.from_pretrained(hf_name)

    @property
    def has_text(self):
        return self.tokenizer is not None

    @torch.no_grad()
    def encode_images(self, images, batch_size=64):
        feats = []
        for i in range(0, len(images), batch_size):
            batch = [_load(x) for x in images[i:i + batch_size]]
            px = self.processor(images=batch, return_tensors="pt")["pixel_values"]
            px = px.to(self.device, dtype=self.dtype)
            if self.kind == "dinov2":
                h = self.model(pixel_values=px).last_hidden_state
                f = torch.cat([h[:, 0], h[:, 1:].mean(dim=1)], dim=-1)   # CLS + mean patch token
            else:
                f = _as_tensor(self.model.get_image_features(pixel_values=px))
            feats.append(F.normalize(f.float(), dim=-1).cpu())
        return torch.cat(feats)

    @torch.no_grad()
    def encode_texts(self, texts, batch_size=64):
        assert self.has_text, f"{self.kind} has no text encoder."
        feats = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i + batch_size]
            if self.kind == "siglip":
                tok = self.tokenizer(chunk, padding="max_length", max_length=64,
                                     truncation=True, return_tensors="pt")
                f = self.model.get_text_features(input_ids=tok["input_ids"].to(self.device))
            else:
                tok = self.tokenizer(chunk, padding=True, max_length=77,
                                     truncation=True, return_tensors="pt")
                f = self.model.get_text_features(
                    input_ids=tok["input_ids"].to(self.device),
                    attention_mask=tok["attention_mask"].to(self.device))
            feats.append(F.normalize(_as_tensor(f).float(), dim=-1).cpu())
        return torch.cat(feats)


def text_prototypes(fm, class_names, templates=PROMPT_TEMPLATES, overrides=None):
    """Zero-shot classifier: mean text embedding over prompt templates, one row per class."""
    protos = []
    for name in class_names:
        phrase = class_to_text(name, overrides)
        feats = fm.encode_texts([t.format(phrase) for t in templates])
        protos.append(F.normalize(feats.mean(0), dim=0))
    return torch.stack(protos)


# ----------------------------------------------------------------------------
# Trainable prototype head
# ----------------------------------------------------------------------------
class PrototypeHead(nn.Module):
    """Cosine-similarity classifier on frozen features. Rows of W are class prototypes."""

    def __init__(self, init_W, scale=100.0):
        super().__init__()
        self.W = nn.Parameter(init_W.clone())
        self.register_buffer("W0", init_W.clone())     # starting point used as an anchor
        self.scale = scale

    def forward(self, feats):
        return self.scale * feats @ F.normalize(self.W, dim=-1).T

    def prototypes(self):
        return F.normalize(self.W.detach(), dim=-1).cpu()


def prototypes_from_views(view_feats):
    """[num_classes, n_views, D] -> [num_classes, D] mean prototype."""
    return F.normalize(view_feats.mean(dim=1), dim=-1)


def fuse(text_proto, image_proto, alpha=0.5):
    """Blend the zero-shot text prototype with the one-shot image prototype."""
    return F.normalize(alpha * text_proto + (1 - alpha) * image_proto, dim=-1)


# ----------------------------------------------------------------------------
# Test-time augmentation and adaptation (use UNLABELED test images only)
# ----------------------------------------------------------------------------
def tta_views(image):
    """Original, flip, and two center crops (each also flipped)."""
    w, h = image.size
    views = [image, T.functional.hflip(image)]
    for frac in (0.9, 0.75):
        cw, ch = int(w * frac), int(h * frac)
        left, top = (w - cw) // 2, (h - ch) // 2
        crop = image.crop((left, top, left + cw, top + ch))
        views += [crop, T.functional.hflip(crop)]
    return views


def encode_images_tta(fm, images, views_fn=tta_views, chunk=32):
    """Average the embeddings of several views of every image."""
    out = []
    for i in range(0, len(images), chunk):
        all_views, counts = [], []
        for x in images[i:i + chunk]:
            v = views_fn(_load(x))
            all_views += v
            counts.append(len(v))
        feats = fm.encode_images(all_views)
        j = 0
        for c in counts:
            out.append(F.normalize(feats[j:j + c].mean(0), dim=0))
            j += c
    return torch.stack(out)


@torch.no_grad()
def sinkhorn(logits, n_iter=3):
    """Soft assignments with equal total mass per class (assumes a roughly balanced pool)."""
    logits = logits.double()
    Q = torch.exp(logits - logits.max())
    for _ in range(n_iter):
        Q = Q / Q.sum(0, keepdim=True)
        Q = Q / Q.sum(1, keepdim=True)
    return Q.float()


@torch.no_grad()
def transductive_refine(X, W0, iters=10, temp=0.03, gamma=0.5, balance=True):
    """Soft k-means in embedding space, anchored to the one-shot prototypes W0."""
    W = W0.clone()
    for _ in range(iters):
        logits = X @ W.T / temp
        Q = sinkhorn(logits) if balance else torch.softmax(logits, dim=1)
        centers = F.normalize(Q.T @ X, dim=-1)
        W = F.normalize((1 - gamma) * W0 + gamma * centers, dim=-1)
    return W


def accuracy(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item() * 100

In [ ]:
import importlib, oneshot_lib
importlib.reload(oneshot_lib)
from oneshot_lib import (
    FoundationModel, PrototypeHead, PROMPT_TEMPLATES, text_prototypes, encode_images_tta,
    prototypes_from_views, fuse, transductive_refine, accuracy
)

## 5. Training and evaluation utilities

`train_head` trains the prototype head on augmented support views and tracks loss, accuracy, and how far the head has drifted from its starting point. There is no labeled validation data in a one-shot setting, so nothing is monitored on the test set during training. `run_stages` evaluates the whole recipe stage by stage, which doubles as the ablation study for the report.

In [ ]:
def train_head(
    model,
    view_feats,
    optimizer,
    scheduler=None,
    epochs=30,
    batch_size=128,
    label_smoothing=0.1,
    lam=10.0,                 # strength of the L2 anchor to the starting prototypes
    device="cpu",
    seed=0,
    verbose=True
):
    """Train the prototype head on augmented views of the 16 support images. view_feats: [C, V, D]."""
    C, V, D = view_feats.shape
    X = view_feats.reshape(C * V, D).to(device)
    y = torch.arange(C).repeat_interleave(V).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    generator = torch.Generator().manual_seed(seed)
    model = model.to(device)

    history = {"train_loss": [], "train_acc": [], "anchor_drift": [], "learning_rate": []}

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X), generator=generator).to(device)
        total_loss, total_correct, total_seen = 0.0, 0, 0

        for i in range(0, len(X), batch_size):
            idx = perm[i:i + batch_size]
            logits = model(X[idx])
            anchor = ((model.W - model.W0) ** 2).sum(-1).mean()
            loss = criterion(logits, y[idx]) + lam * anchor

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(idx)
            total_correct += (logits.argmax(dim=1) == y[idx]).sum().item()
            total_seen += len(idx)

        current_lr = optimizer.param_groups[0]["lr"]
        if scheduler is not None:
            scheduler.step()

        drift = (F.normalize(model.W.detach(), dim=-1) - model.W0).norm(dim=-1).mean().item()
        history["train_loss"].append(total_loss / total_seen)
        history["train_acc"].append(total_correct / total_seen)
        history["anchor_drift"].append(drift)
        history["learning_rate"].append(current_lr)

        if verbose:
            print(
                f"Epoch [{epoch + 1:02d}/{epochs}] | "
                f"Train Loss: {history['train_loss'][-1]:.4f} | "
                f"Train Acc: {history['train_acc'][-1]:.4f} | "
                f"Drift: {drift:.4f} | LR: {current_lr:.6f}"
            )

    return model, history


@torch.inference_mode()
def evaluate(model, feats, labels, device):
    """Loss and accuracy (as a fraction) of a head on precomputed features."""
    model.eval()
    logits = model(feats.to(device))
    labels = labels.to(device)
    loss = nn.CrossEntropyLoss()(logits, labels).item()
    acc = (logits.argmax(dim=1) == labels).float().mean().item()
    return loss, acc


def plot_training_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, history["train_loss"])
    axes[0].set_title("Training loss")

    axes[1].plot(epochs, history["train_acc"])
    axes[1].set_title("Accuracy on augmented support views")

    axes[2].plot(epochs, history["anchor_drift"])
    axes[2].set_title("Drift from starting prototypes")

    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
M1 = "1. Zero-shot (text prompts)"
M2 = "2. One-shot prototype (raw image)"
M3 = "3. + augmented support views"
M4 = "4. + text/image fusion"
M5 = "5. + trained head"
M6 = "6. + test-time augmentation"
M7 = "7. + transductive refinement"
ALL_METHODS = [M1, M2, M3, M4, M5, M6, M7]


def prepare_backbone(backbone_key):
    """Load a frozen backbone and encode everything that does not depend on the support set."""
    kind, hf_name = BACKBONES[backbone_key]
    t0 = time.time()
    fm = FoundationModel(kind, hf_name, device)
    text_proto = text_prototypes(fm, class_names) if fm.has_text else None

    Xc = fm.encode_images(test_paths)                                        # clean test features
    Xt = encode_images_tta(fm, test_paths) if USE_TTA else Xc                # test-time augmented
    Xu = encode_images_tta(fm, unlabeled_paths) if unlabeled_paths else None
    pool = Xt if Xu is None else torch.cat([Xt, Xu])                         # unlabeled features for stage 7
    print(f"[{backbone_key}] features ready in {time.time() - t0:.0f}s")
    return dict(fm=fm, text_proto=text_proto, Xc=Xc, Xt=Xt, pool=pool, backbone=backbone_key)


def run_stages(seed, ctx, verbose=False):
    """Build the support set for `seed`, train the head, and evaluate every stage of the recipe."""
    fm, text_proto, Xc, Xt, pool = ctx["fm"], ctx["text_proto"], ctx["Xc"], ctx["Xt"], ctx["pool"]

    images, labels, files, _ = build_support_set(train_folder, seed)
    support_ds = OneShotSupportDataset(images, labels.tolist(), N_AUG_VIEWS, seed)
    view_feats = fm.encode_images(support_ds.views).reshape(num_classes, N_AUG_VIEWS, -1)

    sup_raw = F.normalize(view_feats[:, 0], dim=-1)           # view 0 = the original image
    sup_aug = prototypes_from_views(view_feats)
    fused = fuse(text_proto, sup_aug, ALPHA) if text_proto is not None else sup_aug

    set_random_seed(seed)
    head = PrototypeHead(fused, scale=HEAD_CFG["scale"])
    optimizer = optim.AdamW(head.parameters(), lr=HEAD_CFG["lr"], weight_decay=0.0)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_CFG["epochs"])
    head, history = train_head(
        head, view_feats, optimizer, scheduler,
        epochs=HEAD_CFG["epochs"], batch_size=HEAD_CFG["batch_size"],
        label_smoothing=HEAD_CFG["label_smoothing"], lam=HEAD_CFG["lam"],
        device=device, seed=seed, verbose=verbose
    )
    W_head = head.prototypes()
    W_ref = transductive_refine(pool, W_head, **REFINE_CFG)

    logits = {}
    if text_proto is not None:
        logits[M1] = Xc @ text_proto.T
    logits[M2] = Xc @ sup_raw.T
    logits[M3] = Xc @ sup_aug.T
    logits[M4] = Xc @ fused.T
    logits[M5] = Xc @ W_head.T
    logits[M6] = Xt @ W_head.T
    logits[M7] = Xt @ W_ref.T

    accs = {m: accuracy(l, test_labels) for m, l in logits.items()}
    return dict(accs=accs, logits=logits, history=history, files=files, W_head=W_head, W_refined=W_ref)


def run_all_seeds(ctx, seeds=SEEDS):
    """Evaluate every stage for several support sets. Returns (results table, per-seed details)."""
    kept = {s: run_stages(s, ctx, verbose=(s == SUPPORT_SEED)) for s in seeds}
    table = pd.DataFrame({s: kept[s]["accs"] for s in seeds}).reindex(ALL_METHODS)
    table["mean"] = table[list(seeds)].mean(axis=1)
    table["std"] = table[list(seeds)].std(axis=1)
    return table, kept

### 5.1 Configure and run an experiment

Choose the backbone and recipe settings, then extract features with the frozen backbone (done once, since the backbone never changes) and train and evaluate the head for each support set.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(
        f"Available GPU memory: "
        f"{torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB"
    )

In [ ]:
set_random_seed(SEED)

# (kind, Hugging Face model id). kind selects how features are extracted.
BACKBONES = {
    "clip_b16":      ("clip",   "openai/clip-vit-base-patch16"),
    "clip_l14":      ("clip",   "openai/clip-vit-large-patch14"),
    "siglip_b16":    ("siglip", "google/siglip-base-patch16-224"),
    "siglip_so400m": ("siglip", "google/siglip-so400m-patch14-384"),
    "dinov2_b":      ("dinov2", "facebook/dinov2-base"),     # image-only: no text prompts
}
BACKBONE = "clip_l14"

# -------------------------
# Recipe configuration
# -------------------------
ALPHA = 0.5               # weight of the text prototype vs the image prototype in the fusion
USE_TTA = True            # test-time augmentation (flip + center crops, averaged)
HEAD_CFG = dict(epochs=30, batch_size=128, lr=2e-4, lam=10.0, scale=100.0, label_smoothing=0.1)
REFINE_CFG = dict(iters=10, temp=0.03, gamma=0.5, balance=True)   # transductive refinement

# Stage whose head is exported and evaluated at the end: "head" (stage 6) or "refined" (stage 7)
FINAL_HEAD = "refined"

# -------------------------
# Load the frozen backbone and extract features
# -------------------------
ctx = prepare_backbone(BACKBONE)

In [ ]:
# -------------------------
# One-shot training and evaluation for every support set
# -------------------------
results_table, kept = run_all_seeds(ctx, SEEDS)
display(results_table.round(2))

In [ ]:
# After training: the training curve of the head for the committed support set
plot_training_history(kept[SUPPORT_SEED]["history"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(results_table.index[::-1], results_table["mean"][::-1],
        xerr=results_table["std"][::-1], capsize=3)
ax.set_xlabel("Test accuracy (%), mean ± std over support sets")
ax.set_title(f"Ablation: {BACKBONE}")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### Display results: confusion matrix and per-class accuracy

Use this for the "observations" part of the report: which classes get confused, and why (for example Street / InsideCity / TallBuilding, or Bedroom / LivingRoom).

In [ ]:
final_stage = M7 if FINAL_HEAD == "refined" else M6
final_logits = kept[SUPPORT_SEED]["logits"][final_stage]
preds = final_logits.argmax(dim=1).numpy()
labels_np = test_labels.numpy()
print(f"{final_stage} | support seed {SUPPORT_SEED} | accuracy: {(preds == labels_np).mean() * 100:.2f}%")

cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(labels_np, preds):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(num_classes):
    for j in range(num_classes):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)
plt.colorbar(im)
plt.tight_layout()
plt.show()

per_class = pd.DataFrame({"Class": class_names, "Accuracy (%)": 100 * cm.diagonal() / cm.sum(axis=1)})
display(per_class.sort_values("Accuracy (%)").round(1))

### 5.2 Compare foundation models (optional)

The assignment asks you to share your observations about foundation models. This cell runs the same pipeline for every backbone, which downloads several models and can take a while, so it is off by default.

In [ ]:
RUN_BACKBONE_SWEEP = False

if RUN_BACKBONE_SWEEP:
    rows = []
    for key in BACKBONES:
        if key == BACKBONE:
            table = results_table
        else:
            other_ctx = prepare_backbone(key)
            table, _ = run_all_seeds(other_ctx, SEEDS)
            del other_ctx
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        rows.append({
            "Backbone": key,
            "Zero-shot": table.loc[M1, "mean"],
            "One-shot prototype": table.loc[M2, "mean"],
            "+ fusion": table.loc[M4, "mean"],
            "+ head + TTA": table.loc[M6, "mean"],
            "+ transductive": table.loc[M7, "mean"],
        })
    sweep_table = pd.DataFrame(rows).set_index("Backbone")
    display(sweep_table.round(2))
else:
    print("Set RUN_BACKBONE_SWEEP = True to compare all backbones.")

## 6. Experiment notes and model improvements

Record backbone and recipe changes. Possible experiments include the backbone (CLIP, SigLIP, DINOv2) and its size, prompt templates, class descriptions, number of augmented views, augmentation strength, `ALPHA`, head learning rate and anchor strength, test-time augmentation, and transductive refinement settings. There is no labeled validation set in one-shot learning, so any comparison here is made on the test set. Keep the number of variants small, fix `SUPPORT_SEED` before looking at scores, report the mean over support sets, and disclose your tuning policy in the report.

## 6.1 Your turn: beat the baseline

The zero-shot result is the baseline. The objective is to substantially improve on it using only the 16 labeled images.

Rather than asking an AI tool to "make the accuracy higher," I formulated hypotheses about possible limitations of the approach and tested them experimentally. Examples of the questions I considered included whether a single image is a noisy prototype, whether text and image information complement each other, whether a larger or differently pretrained backbone would transfer better, whether augmentation helps or hurts, whether the head was overfitting the 16 images, and whether unlabeled test images can correct prototype bias.

For each important experiment, I recorded what I changed, why I changed it, and what I learned from the results. Edit the table below (double-click this cell) and add rows as needed.

| Experiment | What changed? | Why did you try it? | Highest accuracy | What did you learn? |
|---|---|---|---:|---|
| Baseline | Zero-shot text prompts (stage 1) | Reference point with no labeled images | | |
| Experiment 1 | One-shot image prototypes (stage 2) | How far do 16 images alone get us? | | |
| Experiment 2 | Augmented support views and text/image fusion (stages 3-4) | Is a single image a noisy prototype? Do the two modalities complement each other? | | |
| Experiment 3 | Trained head and test-time augmentation (stages 5-6) | Does anchored training beat plain prototypes? | | |
| Experiment 4 | Transductive refinement (stage 7) | Can unlabeled test features fix prototype bias? | | |
| Experiment 5 | Backbone comparison (CLIP vs SigLIP vs DINOv2) | Which pretraining objective transfers best? | | |
| | | | | |
| | | | | |

**Do not pick the support seed by looking at test scores; report the mean over support sets alongside your final number.**

In [ ]:
# Optional helper: print a ready-to-paste markdown row from the latest run
stage = results_table.loc[final_stage]
print(f"| Experiment N | {final_stage} with {BACKBONE} | WHY | "
      f"{stage[SUPPORT_SEED]:.2f} (seed {SUPPORT_SEED}); mean {stage['mean']:.2f} ± {stage['std']:.2f} | WHAT YOU LEARNED |")

## 7. Final test evaluation

Run this only after fixing your final configuration and support seed. Report the mean ± std over support sets together with the single-seed number.

In [ ]:
W_final = kept[SUPPORT_SEED]["W_refined"] if FINAL_HEAD == "refined" else kept[SUPPORT_SEED]["W_head"]
final_head = PrototypeHead(W_final, scale=HEAD_CFG["scale"]).to(device)

test_loss, test_acc = evaluate(final_head, ctx["Xt"], test_labels, device)
print(f"Final test loss: {test_loss:.4f}")
print(f"Final test accuracy: {test_acc:.4f}")

stage = results_table.loc[final_stage]
print(f"Mean ± std over {len(SEEDS)} support sets: {stage['mean']:.2f} ± {stage['std']:.2f}%")

## 8. AI-assisted development checkpoint

If you use an AI coding assistant, document the tools used, representative ways AI assisted, at least one incorrect or ineffective suggestion, how you verified or corrected it, and one important decision you made yourself. Verify tensor shapes, model names, assumptions, and code behavior before relying on generated code.

## 9. GitHub and project requirements

Keep your code and experiment history in GitHub. The report must link a repository that contains the trained model, inference code, and instructions for reproducing the results. A useful repository may contain:

```text
README.md
AI_USAGE.md
notebooks/
    one_shot_template.ipynb
oneshot_lib.py              # backbone, head, test-time adaptation
inference.py                # inference code
oneshot_artifacts.pt        # trained head + configuration
support_set/                # the 16 labeled images that were used
requirements.txt
reproduction instructions
```

Record the final configuration, tuning strategy, test result, important experiments, and failure analysis. Be prepared to explain the foundation model you chose, the prototype head, how the single labeled images are used, and the test-time adaptation. The report (max 2 pages) needs: (1) best test accuracy and the GitHub link, (2) your foundation-model / training recipe (**weighted 3x**), and (3) a conclusion on balancing model complexity, data availability, and compute.

### 9.1 Export for GitHub

The "trained model" is the frozen pretrained backbone (downloaded from Hugging Face by name) plus the small trained head. The cells below save the head and configuration, copy the 16 support images, and write `inference.py`, `README.md`, and `requirements.txt`.

In [ ]:
kind, hf_name = BACKBONES[BACKBONE]

# Copy the 16 labeled images so the repo shows exactly what was used
support_dir = PROJECT_ROOT / "support_set"
support_dir.mkdir(exist_ok=True)
saved_support_files = []
for cls, f in zip(class_names, kept[SUPPORT_SEED]["files"]):
    dst = support_dir / f"{cls}{Path(f).suffix}"
    shutil.copy(f, dst)
    saved_support_files.append(str(dst.relative_to(PROJECT_ROOT)))

artifacts = dict(
    backbone_key=BACKBONE,
    backbone_kind=kind,
    hf_name=hf_name,
    class_names=list(class_names),
    W=W_final.cpu(),                          # [num_classes, D] normalized head
    tta=bool(USE_TTA),
    transductive=(FINAL_HEAD == "refined"),   # stage 7 is applied to the batch of test images
    refine_cfg=dict(REFINE_CFG),
    support_seed=int(SUPPORT_SEED),
    support_files=saved_support_files,
    prompt_templates=list(PROMPT_TEMPLATES),
)
torch.save(artifacts, PROJECT_ROOT / "oneshot_artifacts.pt")
print("Saved oneshot_artifacts.pt | head shape:", tuple(W_final.shape))
print("Support images copied to:", support_dir)

In [ ]:
%%writefile inference.py
"""
One-shot scene classification inference.

Examples:
    python inference.py --image path/to/photo.jpg
    python inference.py --data data/test            # ImageFolder layout; prints accuracy
    python inference.py --data data/test --no-transductive
"""
import argparse

import torch
from torchvision import datasets

from oneshot_lib import FoundationModel, encode_images_tta, transductive_refine, accuracy


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--artifacts", default="oneshot_artifacts.pt")
    ap.add_argument("--image", help="classify a single image file")
    ap.add_argument("--data", help="ImageFolder-style directory; reports accuracy")
    ap.add_argument("--no-transductive", action="store_true",
                    help="skip test-time refinement on the unlabeled batch")
    args = ap.parse_args()
    assert args.image or args.data, "Pass --image or --data."

    art = torch.load(args.artifacts, map_location="cpu", weights_only=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    fm = FoundationModel(art["backbone_kind"], art["hf_name"], device)
    W, names = art["W"], art["class_names"]

    def embed(paths):
        return encode_images_tta(fm, paths) if art["tta"] else fm.encode_images(paths)

    if args.image:
        feats = embed([args.image])
        probs = torch.softmax(100.0 * feats @ W.T, dim=1)[0]
        top = probs.topk(3)
        for p, i in zip(top.values, top.indices):
            print(f"{names[i]:>14s}  {p.item() * 100:5.1f}%")
        return

    folder = datasets.ImageFolder(args.data)
    assert folder.classes == names, "Class folders do not match the trained head."
    paths = [p for p, _ in folder.samples]
    labels = torch.tensor(folder.targets)
    feats = embed(paths)
    if art["transductive"] and not args.no_transductive:
        W = transductive_refine(feats, W, **art["refine_cfg"])
    print(f"Accuracy on {len(paths)} images: {accuracy(feats @ W.T, labels):.2f}%")


if __name__ == "__main__":
    main()

In [ ]:
(PROJECT_ROOT / "requirements.txt").write_text(
    "torch\ntorchvision\ntransformers\nsentencepiece\nprotobuf\npillow\nnumpy\npandas\nmatplotlib\n"
)

readme = f"""# One-shot scene classification (ITCS 6169/8169, Assignment 3)

One labeled training image per class (16 in total) plus the pretrained foundation model
`{hf_name}`. Final test accuracy: **FILL IN** (support seed {SUPPORT_SEED}).

## Contents
- `notebook.ipynb` - the full experiment (rename to match your file)
- `oneshot_lib.py` - backbone, prototype head, test-time augmentation and adaptation
- `inference.py` - run the trained head on an image or an ImageFolder directory
- `oneshot_artifacts.pt` - the trained head (`W`, {tuple(W_final.shape)}) and configuration
- `support_set/` - the 16 labeled training images that were used
- `requirements.txt`

## Reproduce
```bash
pip install -r requirements.txt
# put the dataset at data/train and data/test, then:
python inference.py --data data/test
python inference.py --image some_photo.jpg
```
The backbone weights are downloaded automatically from Hugging Face. To retrain the head,
run the notebook top to bottom (set `SUPPORT_SEED = {SUPPORT_SEED}`).

## Notes
- Only the 16 images in `support_set/` are used with labels.
- Unlabeled test images are used for test-time adaptation (stage 7) when `transductive` is true.
"""
(PROJECT_ROOT / "README.md").write_text(readme)
print("Wrote requirements.txt and README.md")